In [1]:
from ultralytics import YOLO
import torch
import os
from pathlib import Path

print(f"🖥️  Training on: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"💡 Training in 3-epoch chunks - you can stop and resume anytime!\n")

# Configuration
TOTAL_EPOCHS = 50
EPOCHS_PER_SESSION = 3  # Train 3 epochs at a time (~1-2 hours per session)
DATA_PATH = 'F:\\College\\Subject Projects\\VRSU\\Learning\\weapon_data\\data.yaml'
PROJECT_DIR = 'weapon_training'
EXPERIMENT_NAME = 'weapon_detector_v1'

# Check for existing results to determine current epoch
results_csv = Path(PROJECT_DIR) / EXPERIMENT_NAME / 'results.csv'

if results_csv.exists():
    # Read results to get current epoch count
    import pandas as pd
    df = pd.read_csv(results_csv)
    current_epoch = len(df)
    
    print(f"✅ Found existing training")
    print(f"📊 Already completed: {current_epoch} epochs")
    
    if current_epoch >= TOTAL_EPOCHS:
        print(f"\n🎉 Training already complete! You've finished all {TOTAL_EPOCHS} epochs!")
        print(f"📁 Best model: {PROJECT_DIR}/{EXPERIMENT_NAME}/weights/best.pt")
    else:
        next_target = min(current_epoch + EPOCHS_PER_SESSION, TOTAL_EPOCHS)
        print(f"🎯 Will train to epoch {next_target} (adding {next_target - current_epoch} more epochs)\n")
        
        # Load the last checkpoint
        checkpoint_path = Path(PROJECT_DIR) / EXPERIMENT_NAME / 'weights' / 'last.pt'
        model = YOLO(checkpoint_path)
        
        # Train for more epochs (NOT using resume, just continuing)
        results = model.train(
            data=DATA_PATH,
            epochs=next_target,      # Total target epochs
            imgsz=416,              
            batch=8,                
            device='cpu',           
            project=PROJECT_DIR,
            name=EXPERIMENT_NAME,
            patience=50,
            save=True,
            save_period=EPOCHS_PER_SESSION,
            plots=True,
            verbose=True,
            workers=4,              
            cache=False,            
            amp=False,
            resume=False,           # Don't resume, we're starting a new session
            exist_ok=True,
        )
        
        print("\n" + "="*60)
        print("✅ Training session complete!")
        print(f"📊 Completed: {next_target} / {TOTAL_EPOCHS} epochs")
        print(f"📁 Checkpoint saved: {checkpoint_path}")
        print(f"📈 Results: {PROJECT_DIR}/{EXPERIMENT_NAME}/results.png")
        print("\n💡 To continue training:")
        print("   - Just run this cell again!")
        print("="*60)
else:
    print(f"🆕 Starting fresh training with yolov8n.pt\n")
    model = YOLO('yolov8n.pt')
    
    # Train for first session
    results = model.train(
        data=DATA_PATH,
        epochs=EPOCHS_PER_SESSION,
        imgsz=416,              
        batch=8,                
        device='cpu',           
        project=PROJECT_DIR,
        name=EXPERIMENT_NAME,
        patience=50,
        save=True,
        save_period=EPOCHS_PER_SESSION,
        plots=True,
        verbose=True,
        workers=4,              
        cache=False,            
        amp=False,
        exist_ok=True,
    )
    
    print("\n" + "="*60)
    print("✅ First training session complete!")
    print(f"📊 Completed: {EPOCHS_PER_SESSION} / {TOTAL_EPOCHS} epochs")
    print(f"📁 Checkpoint saved: weapon_training/{EXPERIMENT_NAME}/weights/last.pt")
    print("\n💡 To continue training:")
    print("   - Just run this cell again!")
    print("="*60)

🖥️  Training on: CPU
💡 Training in 3-epoch chunks - you can stop and resume anytime!

🆕 Starting fresh training with yolov8n.pt

New https://pypi.org/project/ultralytics/8.4.9 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.243  Python-3.13.2 torch-2.9.1+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=F:\College\Subject Projects\VRSU\Learning\weapon_data\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0

KeyboardInterrupt: 

## 🎯 Training Instructions

**How to use this checkpoint-based training:**

1. **Start Training:** Run the training cell above
2. **Stop Anytime:** After 3 epochs complete (~1-2 hours), you can safely close your laptop
3. **Resume Later:** Just run the same training cell again - it auto-resumes!
4. **Check Progress:** Run the progress cell to see how many epochs done
5. **Repeat:** Keep running until you reach 50 total epochs (about 17 sessions)

**Training Plan:**
- Total epochs needed: 50
- Epochs per session: 3
- Total sessions: 17 sessions (50 ÷ 3 ≈ 17)
- Time per session: ~1-2 hours on CPU

**Each session saves automatically at:**
- `weapon_training/weapon_detector_v1/weights/last.pt` (latest checkpoint)
- `weapon_training/weapon_detector_v1/weights/best.pt` (best model so far)

In [2]:
import pandas as pd
from pathlib import Path

# Check training progress
results_csv = Path('weapon_training/weapon_detector_v1/results.csv')

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df = df.strip()  # Remove whitespace from column names
    
    total_epochs = len(df)
    print(f"📈 Training Progress: {total_epochs} / 50 epochs completed\n")
    
    # Show last 5 epochs performance
    print("Last 5 Epochs Performance:")
    print("="*60)
    
    cols_to_show = ['epoch', 'train/box_loss', 'train/cls_loss', 'metrics/mAP50(B)']
    available_cols = [col for col in cols_to_show if col in df.columns]
    
    if available_cols:
        print(df[available_cols].tail(5).to_string(index=False))
    else:
        print(df.tail(5))
    
    print("\n" + "="*60)
    print(f"🎯 Sessions remaining: {(50 - total_epochs) // 3} (3 epochs each)")
    
else:
    print("⚠️  No training results found yet. Run the training cell first!")

AttributeError: 'DataFrame' object has no attribute 'strip'

## 📊 Check Training Progress

Run this cell to see how many epochs you've completed so far:

## Validate Best Model - Check Accuracy

In [ ]:
from ultralytics import YOLO

# Load best.pt model to validate
model_path = r"F:\College\Subject Projects\VRSU\Learning\weapon_training\weapon_detector_v1\weights\best.pt"
model_to_validate = YOLO(model_path)

print("🔍 Validating best.pt model on validation dataset...")
print(f"Model: {model_path}\n")

# Run validation
metrics = model_to_validate.val(data=DATA_PATH, split='val')

# Class names mapping
class_names = ['Grenade', 'Knife', 'Pistol', 'Rifle']

# Display key metrics
print("\n" + "="*60)
print("📊 ACCURACY METRICS")
print("="*60)
print(f"mAP50-95 (Mean Average Precision): {metrics.box.map:.4f}")
print(f"mAP50 (at IoU=0.50):               {metrics.box.map50:.4f}")
print(f"mAP75 (at IoU=0.75):               {metrics.box.map75:.4f}")
print(f"Precision:                         {metrics.box.mp:.4f}")
print(f"Recall:                            {metrics.box.mr:.4f}")
print("="*60)

# Per-class metrics if available
if hasattr(metrics.box, 'maps') and len(metrics.box.maps) > 0:
    print("\n📋 Per-Class mAP50-95:")
    for i, map_value in enumerate(metrics.box.maps):
        class_name = class_names[i] if i < len(class_names) else f"Class {i}"
        emoji = "🎯" if i == 0 else "🔪" if i == 1 else "🔫"
        print(f"  {emoji} {class_name:8s}: {map_value:.4f} ({map_value*100:.2f}%)")
    
    # Highlight weakest class
    min_idx = metrics.box.maps.argmin()
    print(f"\n⚠️  Weakest detection: {class_names[min_idx]} at {metrics.box.maps[min_idx]*100:.2f}%")
    print(f"💡 Extended training recommended to improve {class_names[min_idx]} detection")


🔍 Validating best.pt model on validation dataset...
Model: F:\College\Subject Projects\VRSU\Learning\weapon_training\weapon_detector_v1\weights\best.pt

Ultralytics 8.3.243  Python-3.13.2 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 105.341.3 MB/s, size: 33.6 KB)
val: Scanning F:\College\Subject Projects\VRSU\Learning\weapon_data\valid\labels.cache... 1895 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1895/1895 964.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 119/119 7.1it/s 16.7s0.1s
                   all       1895       3052      0.844      0.737      0.802      0.584
               Grenade        321        545      0.909      0.808      0.868      0.724
                 Knife        244        345       0.76      0.661      0.708      0.474
                Pis

: 

## 🔄 Continue Training from Best Checkpoint

In [11]:
from ultralytics import YOLO

# Load the best model checkpoint
best_model_path = r"F:\College\Subject Projects\VRSU\Learning\weapon_training\weapon_detector_v1\weights\best.pt"
model_continue = YOLO(best_model_path)

# Set number of additional epochs to train
ADDITIONAL_EPOCHS = 10  # Adjust this number as needed

print(f"🚀 Resuming training from best checkpoint...")
print(f"📁 Model: {best_model_path}")
print(f"📊 Training for {ADDITIONAL_EPOCHS} more epochs\n")

# Continue training
results = model_continue.train(
    data=DATA_PATH,
    epochs=ADDITIONAL_EPOCHS,
    imgsz=416,
    batch=8,
    device='cpu',
    project=PROJECT_DIR,
    name=EXPERIMENT_NAME,
    exist_ok=True,
    patience=50,
    save=True,
    save_period=3,
    plots=True,
    verbose=True
)

print("\n✅ Training completed!")
print(f"📁 Results saved to: {results.save_dir}")

🚀 Resuming training from best checkpoint...
📁 Model: F:\College\Subject Projects\VRSU\Learning\weapon_training\weapon_detector_v1\weights\best.pt
📊 Training for 10 more epochs

New https://pypi.org/project/ultralytics/8.4.9 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.243  Python-3.13.2 torch-2.9.1+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=F:\College\Subject Projects\VRSU\Learning\weapon_data\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.

KeyboardInterrupt: 

## 🎯 Extended Training - Push to 100 Epochs for Better Knife Detection

In [1]:
from ultralytics import YOLO
import pandas as pd
from pathlib import Path

# Configuration for extended training
DATA_PATH = 'F:\\College\\Subject Projects\\VRSU\\Learning\\weapon_data\\data.yaml'
PROJECT_DIR = 'weapon_training'
EXPERIMENT_NAME = 'weapon_detector_v1'
EXTENDED_TARGET = 100  # New target: 100 total epochs

# Check current progress
results_csv = Path(PROJECT_DIR) / EXPERIMENT_NAME / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    current_epoch = len(df)
else:
    current_epoch = 0

additional_epochs = EXTENDED_TARGET - current_epoch

print("="*60)
print("🎯 EXTENDED TRAINING - IMPROVING KNIFE DETECTION")
print("="*60)
print(f"📊 Current progress: {current_epoch} epochs completed")
print(f"🎯 Extended target: {EXTENDED_TARGET} total epochs")
print(f"⏳ Will train {additional_epochs} more epochs")
print(f"💡 Focus: Improve knife detection (currently 44%)")
print("="*60)

if additional_epochs <= 0:
    print("\n✅ Already reached 100 epochs!")
else:
    # Load your current best model
    best_model_path = r"F:\College\Subject Projects\VRSU\Learning\weapon_training\weapon_detector_v1\weights\best.pt"
    model = YOLO(best_model_path)
    
    print(f"\n🚀 Loading best model from 50 epochs...")
    print(f"📁 Model: {best_model_path}")
    print(f"📈 Current mAP50-95: 57.02%")
    print(f"🔪 Knife detection: 44.35% → Target: 50%+\n")
    
    # Continue training for 50 more epochs
    results = model.train(
        data=DATA_PATH,
        epochs=additional_epochs,  # Train for 50 more epochs
        imgsz=416,
        batch=16,
        device='cuda',
        project=PROJECT_DIR,
        name=EXPERIMENT_NAME,
        exist_ok=True,
        patience=100,  # Increased patience for longer training
        save=True,
        save_period=5,
        plots=True,
        verbose=True,
        workers=4,
        cache=False,
        amp=False
    )
    
    print("\n" + "="*60)
    print("✅ Extended Training to 100 Epochs Completed!")
    print(f"📁 Best model: {PROJECT_DIR}/{EXPERIMENT_NAME}/weights/best.pt")
    print(f"📈 Results: {results.save_dir}")
    print("\n💡 Run the validation cell (Cell 6) to check improved accuracy!")
    print("="*60)

🎯 EXTENDED TRAINING - IMPROVING KNIFE DETECTION
📊 Current progress: 31 epochs completed
🎯 Extended target: 100 total epochs
⏳ Will train 69 more epochs
💡 Focus: Improve knife detection (currently 44%)

🚀 Loading best model from 50 epochs...
📁 Model: F:\College\Subject Projects\VRSU\Learning\weapon_training\weapon_detector_v1\weights\best.pt
📈 Current mAP50-95: 57.02%
🔪 Knife detection: 44.35% → Target: 50%+

New https://pypi.org/project/ultralytics/8.4.11 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.243  Python-3.13.2 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=F:\College\Subject Projects\VRSU\Learning\weapon_data\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dn